# Notebook 07 — Reviews & Sentiment Analysis

**Amazon Market Intelligence**  
**Questions answered:** Q10 (What are customers actually saying?), Q11 (Can I trust these reviews?)  
**Tool modes served:** Voice of Customer, Review Trust Score  
**Gold tables:** `gold_review_sentiment`, `gold_review_keywords`, `gold_product_review_summary`, `gold_review_trust`, `gold_product_review_trust`, `gold_temporal_trends`  

Prior findings to validate:  
- Finding #35: Ratings declining 4.28 → 4.02 (2019–2022)  
- Finding #40: Subscription Boxes = 25.1% negative (highest)  
- Finding #41: Negative reviews are longer than positive  
- Finding #42: Unverified 5-star gap is NEGATIVE across all categories  
- Finding #51: Clothing categories ~1% review match rate

## 0 — Setup

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

DB_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'amazon_intelligence.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)
TEMPLATE = 'plotly_white'
CHARTS_DIR = 'charts/07_reviews_sentiment'

def save_chart(fig, name, folder=CHARTS_DIR):
    os.makedirs(folder, exist_ok=True)
    fig.write_html(f'{folder}/{name}.html')
    try:
        fig.write_image(f'{folder}/{name}.png', width=1200, height=700, scale=2)
    except Exception as e:
        print(f'PNG failed: {e}')

print(f'Connected to: {DB_PATH}')

Connected to: c:\Users\thinkpad\Desktop\amazon-market-intelligence\data\amazon_intelligence.duckdb


## 1 — Schema Discovery

In [2]:
tables = [
    'gold_review_sentiment',
    'gold_review_keywords',
    'gold_product_review_summary',
    'gold_review_trust',
    'gold_product_review_trust',
    'gold_temporal_trends'
]

for t in tables:
    print(f'\n{"=" * 50}')
    print(f'=== {t} ===')
    print(con.sql(f'DESCRIBE {t}').df().to_string())
    print(f'Rows: {con.sql(f"SELECT COUNT(*) FROM {t}").fetchone()[0]:,}')
    con.sql(f'SELECT * FROM {t} LIMIT 3').show()


=== gold_review_sentiment ===
                 column_name column_type null   key default extra
0            source_category     VARCHAR  YES  None    None  None
1              total_reviews      BIGINT  YES  None    None  None
2                 avg_rating      DOUBLE  YES  None    None  None
3              median_rating      DOUBLE  YES  None    None  None
4             positive_count     HUGEINT  YES  None    None  None
5              neutral_count     HUGEINT  YES  None    None  None
6             negative_count     HUGEINT  YES  None    None  None
7               pct_positive      DOUBLE  YES  None    None  None
8                pct_neutral      DOUBLE  YES  None    None  None
9               pct_negative      DOUBLE  YES  None    None  None
10             pct_with_text      DOUBLE  YES  None    None  None
11           avg_text_length      DOUBLE  YES  None    None  None
12  avg_text_length_negative      DOUBLE  YES  None    None  None
13  avg_text_length_positive      DOUBLE  YES

## 2 — Load Gold Tables


In [3]:
df_sent = con.sql('SELECT * FROM gold_review_sentiment').df()
df_kw = con.sql('SELECT * FROM gold_review_keywords').df()
df_summary = con.sql('SELECT * FROM gold_product_review_summary').df()
df_trust_cat = con.sql('SELECT * FROM gold_review_trust').df()
df_trust_prod = con.sql('SELECT * FROM gold_product_review_trust').df()
df_temporal = con.sql('SELECT * FROM gold_temporal_trends').df()

print(f'Review sentiment (category-level): {len(df_sent):,} rows')
print(f'Review keywords: {len(df_kw):,} rows')
print(f'Product review summary: {len(df_summary):,} rows')
print(f'Review trust (category-level): {len(df_trust_cat):,} rows')
print(f'Review trust (product-level): {len(df_trust_prod):,} rows')
print(f'Temporal trends: {len(df_temporal):,} rows')

Review sentiment (category-level): 33 rows
Review keywords: 165,858 rows
Product review summary: 419,016 rows
Review trust (category-level): 33 rows
Review trust (product-level): 419,016 rows
Temporal trends: 8,721 rows


In [4]:
SENT_CAT_COL = 'source_category'            
SENT_AVG_RATING = 'avg_rating'              
SENT_PCT_NEG = 'pct_negative'                
SENT_AVG_LENGTH = 'avg_text_length'          
SENT_REVIEW_COUNT = 'total_reviews'          
SENT_NEG_LENGTH = 'avg_text_length_negative'  
SENT_POS_LENGTH = 'avg_text_length_positive'   
SENT_VERBOSITY = 'negative_verbosity_ratio'   

KW_CAT_COL = 'source_category'         
KW_WORD = 'word'                       
KW_SENTIMENT = 'sentiment'             
KW_COUNT = 'word_count'                

TRUST_CAT_COL = 'source_category'      
TRUST_VERIFIED_PCT = 'pct_verified'    
TRUST_IMAGE_PCT = 'pct_with_images'   
TRUST_HELPFUL = 'avg_helpful_votes'     
TRUST_5STAR_UNVERIFIED = 'pct_5star_among_unverified'  
TRUST_5STAR_VERIFIED = 'pct_5star_among_verified'      

PSUM_ASIN = 'parent_asin'             
PSUM_REVIEW_COUNT = 'review_count'    
PSUM_AVG_RATING = 'avg_rating'        
PSUM_AVG_LENGTH = 'avg_text_length'   

PTRUST_ASIN = 'parent_asin'           
PTRUST_VERIFIED = 'pct_verified'       

checks = [
    ('df_sent', df_sent, [SENT_CAT_COL, SENT_AVG_RATING, SENT_PCT_NEG, SENT_REVIEW_COUNT, SENT_VERBOSITY]),
    ('df_kw', df_kw, [KW_CAT_COL, KW_WORD, KW_SENTIMENT, KW_COUNT]),
    ('df_trust_cat', df_trust_cat, [TRUST_CAT_COL, TRUST_VERIFIED_PCT, TRUST_5STAR_UNVERIFIED, TRUST_5STAR_VERIFIED]),
    ('df_summary', df_summary, [PSUM_ASIN, PSUM_REVIEW_COUNT, PSUM_AVG_RATING]),
    ('df_trust_prod', df_trust_prod, [PTRUST_ASIN, PTRUST_VERIFIED])
]

for name, df_check, cols in checks:
    for col in cols:
        status = '✅' if col in df_check.columns else '❌ NOT FOUND'
        print(f'{name}.{col} — {status}')

df_sent.source_category — ✅
df_sent.avg_rating — ✅
df_sent.pct_negative — ✅
df_sent.total_reviews — ✅
df_sent.negative_verbosity_ratio — ✅
df_kw.source_category — ✅
df_kw.word — ✅
df_kw.sentiment — ✅
df_kw.word_count — ✅
df_trust_cat.source_category — ✅
df_trust_cat.pct_verified — ✅
df_trust_cat.pct_5star_among_unverified — ✅
df_trust_cat.pct_5star_among_verified — ✅
df_summary.parent_asin — ✅
df_summary.review_count — ✅
df_summary.avg_rating — ✅
df_trust_prod.parent_asin — ✅
df_trust_prod.pct_verified — ✅


---

## 3 — Sentiment Landscape

Finding #40: Subscription Boxes = 25.1% negative reviews (highest).  
Finding #41: Negative reviews are longer than positive.  
Which categories have the angriest customers?

### 3.1 — Negative Review Rate by Category

In [5]:
sent_sorted = df_sent.sort_values(SENT_PCT_NEG, ascending=True)

colors = ['#F44336' if x > 20 else '#FF9800' if x > 15 else '#4CAF50' for x in sent_sorted[SENT_PCT_NEG]]

fig = go.Figure(go.Bar(
    x=sent_sorted[SENT_PCT_NEG],
    y=sent_sorted[SENT_CAT_COL],
    orientation='h',
    marker_color=colors,
    text=[f'{v:.1f}%' for v in sent_sorted[SENT_PCT_NEG]],
    textposition='outside'
))
fig.update_layout(
    title='Negative Review Rate by Category — Where Customers Complain Most',
    xaxis_title='% Negative Reviews (1-2 stars)',
    template=TEMPLATE, height=900
)
save_chart(fig, '01_negative_rate_by_category')
fig.show()

### 3.2 — Average Rating vs Review Volume

In [6]:
fig = px.scatter(
    df_sent,
    x=SENT_REVIEW_COUNT,
    y=SENT_AVG_RATING,
    size=SENT_PCT_NEG,
    hover_name=SENT_CAT_COL,
    title='Category Rating vs Review Volume — Bigger Categories, Harsher Reviews?',
    labels={SENT_REVIEW_COUNT: 'Total Reviews', SENT_AVG_RATING: 'Avg Rating',
            SENT_PCT_NEG: '% Negative'},
    template=TEMPLATE,
    log_x=True
)
fig.update_layout(height=600)
save_chart(fig, '02_rating_vs_volume')
fig.show()

### 3.3 — Review Length: Negative vs Positive

In [7]:
verb_sorted = df_sent.sort_values(SENT_VERBOSITY, ascending=True)

fig = go.Figure(go.Bar(
    x=verb_sorted[SENT_VERBOSITY],
    y=verb_sorted[SENT_CAT_COL],
    orientation='h',
    marker_color=['#F44336' if x > 1.3 else '#FF9800' if x > 1.1 else '#4CAF50' for x in verb_sorted[SENT_VERBOSITY]],
    text=[f'{v:.2f}×' for v in verb_sorted[SENT_VERBOSITY]],
    textposition='outside'
))
fig.add_vline(x=1.0, line_color='black', line_width=2, line_dash='dash',
              annotation_text='Equal length')
fig.update_layout(
    title='Negative Review Verbosity Ratio — Unhappy Customers Write More',
    xaxis_title='Negative ÷ Positive Review Length (>1 = negative is longer)',
    template=TEMPLATE, height=900
)
save_chart(fig, '03_verbosity_ratio')
fig.show()

---

## 4 — Keyword Analysis

What words dominate positive vs negative reviews? This feeds the Voice of Customer tool mode.

### 4.1 — Top Keywords by Sentiment (Overall)

In [13]:
neg_kw = df_kw[df_kw[KW_SENTIMENT] == 'negative'][[KW_CAT_COL, KW_WORD, KW_COUNT]].rename(columns={KW_COUNT: 'neg_count'})
pos_kw = df_kw[df_kw[KW_SENTIMENT] == 'positive'][[KW_CAT_COL, KW_WORD, KW_COUNT]].rename(columns={KW_COUNT: 'pos_count'})

kw_merged = neg_kw.merge(pos_kw, on=[KW_CAT_COL, KW_WORD], how='left')
kw_merged['pos_count'] = kw_merged['pos_count'].fillna(1)
kw_merged['neg_ratio'] = kw_merged['neg_count'] / kw_merged['pos_count']

kw_merged = kw_merged[kw_merged['neg_count'] >= 100]

overall_neg = (
    kw_merged.groupby(KW_WORD)
    .agg(total_neg=('neg_count', 'sum'), total_pos=('pos_count', 'sum'))
    .reset_index()
)
overall_neg['neg_ratio'] = overall_neg['total_neg'] / overall_neg['total_pos'].replace(0, 1)
overall_neg = overall_neg[overall_neg['total_neg'] >= 500]

print('Top 15 complaint-distinctive keywords (high neg/pos ratio + volume):')
top_distinctive = overall_neg.nlargest(15, 'neg_ratio')
for _, row in top_distinctive.iterrows():
    print(f'  {row[KW_WORD]}: {row["neg_ratio"]:.1f}× more in negative, {row["total_neg"]:,} uses')

Top 15 complaint-distinctive keywords (high neg/pos ratio + volume):
  inedible: 699.0× more in negative, 699 uses
  flavorless: 608.0× more in negative, 608 uses
  rancid: 396.5× more in negative, 1,586 uses
  dnf: 383.0× more in negative, 766 uses
  dissatisfied: 335.9× more in negative, 5,375 uses
  unreadable: 326.0× more in negative, 652 uses
  yuk: 305.0× more in negative, 610 uses
  rotten: 292.2× more in negative, 1,169 uses
  watered: 271.7× more in negative, 815 uses
  returnable: 271.1× more in negative, 2,169 uses
  ineffective: 267.1× more in negative, 2,137 uses
  pos: 259.5× more in negative, 2,855 uses
  shoddy: 247.8× more in negative, 991 uses
  yawn: 233.3× more in negative, 700 uses
  unsafe: 225.8× more in negative, 2,032 uses


In [14]:
top_cats = df_sent.nlargest(9, SENT_REVIEW_COUNT)[SENT_CAT_COL].tolist()
print(f'Top 9 categories by review volume: {top_cats}')

fig = make_subplots(rows=3, cols=3, subplot_titles=top_cats,
                    horizontal_spacing=0.22, vertical_spacing=0.12)

for idx, cat in enumerate(top_cats):
    row, col = idx // 3 + 1, idx % 3 + 1
    cat_data = kw_merged[kw_merged[KW_CAT_COL] == cat]
    top10 = cat_data.nlargest(10, 'neg_ratio').sort_values('neg_ratio')

    fig.add_trace(go.Bar(
        x=top10['neg_ratio'], y=top10[KW_WORD],
        orientation='h', marker_color='#F44336',
        text=[f'{v:.1f}×' for v in top10['neg_ratio']],
        textposition='outside',
        showlegend=False
    ), row=row, col=col)

fig.update_annotations(font_size=10)
fig.update_layout(title='Category Complaint Signatures — What Goes Wrong WHERE',
                  template=TEMPLATE, height=1200, width=1400)
save_chart(fig, '05_complaint_keywords_by_category')
fig.show()

Top 9 categories by review volume: ['Home_and_Kitchen', 'Clothing_Shoes_and_Jewelry', 'Electronics', 'Books', 'Tools_and_Home_Improvement', 'Health_and_Household', 'Kindle_Store', 'Beauty_and_Personal_Care', 'Cell_Phones_and_Accessories']


### 4.2 — Category-Specific Keyword Comparison

In [16]:
top_cats = df_sent.nlargest(9, SENT_REVIEW_COUNT)[SENT_CAT_COL].tolist()
print(f'Top 9 categories by review volume: {top_cats}')

fig = make_subplots(rows=3, cols=3, subplot_titles=top_cats,
                    horizontal_spacing=0.22, vertical_spacing=0.12)

for idx, cat in enumerate(top_cats):
    row, col = idx // 3 + 1, idx % 3 + 1
    cat_data = kw_merged[(kw_merged[KW_CAT_COL] == cat) & (kw_merged['neg_ratio'] > 2.0)]
    top10 = cat_data.nlargest(10, 'neg_ratio').sort_values('neg_ratio')

    fig.add_trace(go.Bar(
        x=top10['neg_ratio'], y=top10[KW_WORD],
        orientation='h', marker_color='#7B1FA2',
        text=[f'{v:.1f}×' for v in top10['neg_ratio']],
        textposition='outside',
        showlegend=False
    ), row=row, col=col)

fig.update_annotations(font_size=10)
fig.update_layout(title='Category Complaint Signatures — What Goes Wrong WHERE (ratio > 2×)',
                  template=TEMPLATE, height=1200, width=1400)
save_chart(fig, '05_complaint_keywords_by_category')
fig.show()

Top 9 categories by review volume: ['Home_and_Kitchen', 'Clothing_Shoes_and_Jewelry', 'Electronics', 'Books', 'Tools_and_Home_Improvement', 'Health_and_Household', 'Kindle_Store', 'Beauty_and_Personal_Care', 'Cell_Phones_and_Accessories']


### 4.3 — Positive-Distinctive Keywords
What words signal happy customers? The flip side — words disproportionately used in positive reviews.

In [17]:
pos_main = df_kw[df_kw[KW_SENTIMENT] == 'positive'][[KW_CAT_COL, KW_WORD, KW_COUNT]].rename(columns={KW_COUNT: 'pos_count'})
neg_main = df_kw[df_kw[KW_SENTIMENT] == 'negative'][[KW_CAT_COL, KW_WORD, KW_COUNT]].rename(columns={KW_COUNT: 'neg_count'})

kw_pos_merged = pos_main.merge(neg_main, on=[KW_CAT_COL, KW_WORD], how='left')
kw_pos_merged['neg_count'] = kw_pos_merged['neg_count'].fillna(1)
kw_pos_merged['pos_ratio'] = kw_pos_merged['pos_count'] / kw_pos_merged['neg_count']
kw_pos_merged = kw_pos_merged[kw_pos_merged['pos_count'] >= 500]

overall_pos = (
    kw_pos_merged.groupby(KW_WORD)
    .agg(total_pos=('pos_count', 'sum'), total_neg=('neg_count', 'sum'))
    .reset_index()
)
overall_pos['pos_ratio'] = overall_pos['total_pos'] / overall_pos['total_neg'].replace(0, 1)
overall_pos = overall_pos[overall_pos['total_pos'] >= 500]

print('Top 15 praise-distinctive keywords (high pos/neg ratio + volume):')
top_pos = overall_pos.nlargest(15, 'pos_ratio')
for _, row in top_pos.iterrows():
    print(f'  {row[KW_WORD]}: {row["pos_ratio"]:.1f}× more in positive, {row["total_pos"]:,} uses')

Top 15 praise-distinctive keywords (high pos/neg ratio + volume):
  inspiring: 4228.7× more in positive, 12,686 uses
  captivating: 3850.7× more in positive, 11,552 uses
  steamy: 3727.0× more in positive, 3,727 uses
  yum: 3555.4× more in positive, 17,777 uses
  insightful: 3323.0× more in positive, 6,646 uses
  hilarious: 3100.2× more in positive, 12,401 uses
  excelente: 3089.2× more in positive, 67,963 uses
  gripping: 2998.7× more in positive, 8,996 uses
  journey: 2987.0× more in positive, 8,961 uses
  compliments: 2922.0× more in positive, 14,610 uses
  provoking: 2897.3× more in positive, 8,692 uses
  thrilling: 2839.0× more in positive, 5,678 uses
  suspenseful: 2756.7× more in positive, 8,270 uses
  addition: 2750.4× more in positive, 60,509 uses
  emotional: 2721.7× more in positive, 8,165 uses


In [18]:
top20_pos = overall_pos.nlargest(20, 'pos_ratio').sort_values('pos_ratio')

fig = go.Figure(go.Bar(
    x=top20_pos['pos_ratio'],
    y=top20_pos[KW_WORD],
    orientation='h',
    marker_color='#2E7D32',
    text=[f'{v:.1f}×' for v in top20_pos['pos_ratio']],
    textposition='outside'
))
fig.add_vline(x=1.0, line_color='black', line_width=2, line_dash='dash',
              annotation_text='Equal usage')
fig.update_layout(
    title='Praise-Distinctive Keywords — Words That Signal Happy Customers',
    xaxis_title='Positive ÷ Negative Usage Ratio (higher = more praise-specific)',
    template=TEMPLATE, height=700
)
save_chart(fig, '06_distinctive_positive_keywords')
fig.show()

In [20]:
top_cats = df_sent.nlargest(9, SENT_REVIEW_COUNT)[SENT_CAT_COL].tolist()

fig = make_subplots(rows=3, cols=3, subplot_titles=top_cats,
                    horizontal_spacing=0.22, vertical_spacing=0.12)

for idx, cat in enumerate(top_cats):
    row, col = idx // 3 + 1, idx % 3 + 1
    cat_data = kw_pos_merged[(kw_pos_merged[KW_CAT_COL] == cat) & (kw_pos_merged['pos_ratio'] > 2.0)]
    top10 = cat_data.nlargest(10, 'pos_ratio').sort_values('pos_ratio')

    fig.add_trace(go.Bar(
        x=top10['pos_ratio'], y=top10[KW_WORD],
        orientation='h', marker_color='#1565C0',
        text=[f'{v:.1f}×' for v in top10['pos_ratio']],
        textposition='outside',
        showlegend=False
    ), row=row, col=col)

fig.update_annotations(font_size=10)
fig.update_layout(title='Category Praise Signatures — What Goes RIGHT Where (ratio > 2×)',
                  template=TEMPLATE, height=1200, width=1400)
save_chart(fig, '07_praise_keywords_by_category')
fig.show()

---

## 5 — Review Trust Signals

Finding #42: Unverified 5-star gap is NEGATIVE — unverified reviews are *less* positive than verified.  
This challenges the "fake review" narrative.

### 5.1 — Verified Purchase Rate by Category

In [21]:
trust_sorted = df_trust_cat.sort_values(TRUST_VERIFIED_PCT, ascending=True)

colors = ['#F44336' if x < 60 else '#FF9800' if x < 75 else '#4CAF50' for x in trust_sorted[TRUST_VERIFIED_PCT]]

fig = go.Figure(go.Bar(
    x=trust_sorted[TRUST_VERIFIED_PCT],
    y=trust_sorted[TRUST_CAT_COL],
    orientation='h',
    marker_color=colors,
    text=[f'{v:.1f}%' for v in trust_sorted[TRUST_VERIFIED_PCT]],
    textposition='outside'
))
fig.update_layout(
    title='Verified Purchase Rate by Category — Where Reviews Are Most Trustworthy',
    xaxis_title='% Verified Purchases',
    template=TEMPLATE, height=900
)
save_chart(fig, '06_verified_rate_by_category')
fig.show()

### 5.2 — Trust Signal Heatmap

In [22]:
trust_cols = [c for c in df_trust_cat.columns if c != TRUST_CAT_COL and df_trust_cat[c].dtype in ['float64', 'int64']]
print(f'Available trust signals: {trust_cols}')

trust_matrix = df_trust_cat.set_index(TRUST_CAT_COL)[trust_cols]

trust_norm = (trust_matrix - trust_matrix.min()) / (trust_matrix.max() - trust_matrix.min())

fig = px.imshow(
    trust_norm,
    aspect='auto',
    title='Review Trust Signals by Category (Normalized)',
    labels=dict(color='Normalized Score'),
    color_continuous_scale='RdYlGn',
    template=TEMPLATE
)
fig.update_layout(height=900)
save_chart(fig, '07_trust_signal_heatmap')
fig.show()

Available trust signals: ['total_reviews', 'unique_reviewers', 'reviews_per_reviewer', 'pct_verified', 'pct_unverified', 'pct_with_images', 'pct_with_text', 'avg_helpful_votes', 'avg_text_length', 'pct_5star', 'pct_4star', 'pct_3star', 'pct_2star', 'pct_1star', 'pct_extreme_ratings', 'pct_5star_among_unverified', 'pct_5star_among_verified']


### 5.3 — Verified vs Unverified Rating Gap

In [23]:
df_trust_cat['unverified_5star_gap'] = (
    df_trust_cat[TRUST_5STAR_UNVERIFIED] - df_trust_cat[TRUST_5STAR_VERIFIED]
)

gap_sorted = df_trust_cat.sort_values('unverified_5star_gap')
colors = ['#F44336' if x < 0 else '#4CAF50' for x in gap_sorted['unverified_5star_gap']]

fig = go.Figure(go.Bar(
    x=gap_sorted['unverified_5star_gap'],
    y=gap_sorted[TRUST_CAT_COL],
    orientation='h',
    marker_color=colors,
    text=[f'{v:+.1f}pp' for v in gap_sorted['unverified_5star_gap']],
    textposition='outside'
))
fig.add_vline(x=0, line_color='black', line_width=2)
fig.update_layout(
    title='Unverified vs Verified 5-Star Gap — Are Fake Reviews Inflating Ratings?',
    xaxis_title='Gap in percentage points (negative = unverified LESS positive)',
    template=TEMPLATE, height=900
)
save_chart(fig, '08_verified_unverified_gap')
fig.show()

---

## 6 — Product-Level Review Patterns

Do more reviews = more revenue? What's the review threshold for traction?

### 6.1 — Review Count Distribution

In [25]:
print(f'Products with reviews: {len(df_summary):,}')
print(f'Median reviews/product: {df_summary[PSUM_REVIEW_COUNT].median():.0f}')
print(f'Mean reviews/product: {df_summary[PSUM_REVIEW_COUNT].mean():.0f}')
print(f'Max reviews: {df_summary[PSUM_REVIEW_COUNT].max():,}')

fig = px.histogram(
    df_summary[df_summary[PSUM_REVIEW_COUNT] <= df_summary[PSUM_REVIEW_COUNT].quantile(0.95)],
    x=PSUM_REVIEW_COUNT,
    nbins=50,
    title='Review Count Distribution per Product (95th percentile cutoff)',
    labels={PSUM_REVIEW_COUNT: 'Number of Reviews'},
    template=TEMPLATE
)
fig.update_layout(yaxis_title='Number of Products', height=500)
save_chart(fig, '09_review_count_distribution')
fig.show()

Products with reviews: 419,016
Median reviews/product: 12
Mean reviews/product: 69
Max reviews: 43,237


### 6.2 — Product Review Trust Distribution

In [26]:
print(f'Product trust table: {len(df_trust_prod):,} rows')
print(f'Columns: {df_trust_prod.columns.tolist()}')

if PTRUST_VERIFIED in df_trust_prod.columns:
    fig = px.histogram(
        df_trust_prod[df_trust_prod[PTRUST_VERIFIED] > 0],
        x=PTRUST_VERIFIED,
        nbins=50,
        title='Verified Purchase Rate Distribution (Product-Level)',
        labels={PTRUST_VERIFIED: '% Verified Purchases'},
        template=TEMPLATE
    )
    fig.add_vline(x=df_trust_prod[PTRUST_VERIFIED].median(), line_dash='dash', line_color='red',
                  annotation_text=f'Median: {df_trust_prod[PTRUST_VERIFIED].median():.1f}%')
    fig.update_layout(yaxis_title='Number of Products', height=500)
    save_chart(fig, '10_product_verified_distribution')
    fig.show()
else:
    print(f'{PTRUST_VERIFIED} not found. FIX ME.')

Product trust table: 419,016 rows
Columns: ['parent_asin', 'source_category', 'review_count', 'unique_reviewers', 'reviews_per_reviewer', 'pct_extreme_ratings', 'pct_5star', 'pct_1star', 'pct_verified', 'pct_5star_unverified', 'pct_5star_verified', 'pct_with_text', 'avg_text_length', 'pct_very_short_text', 'reviews_per_day', 'pct_with_helpful_votes']


### 6.3 — Review Count Buckets vs Average Rating

In [27]:
bins = [0, 5, 20, 100, 500, float('inf')]
labels = ['1-5', '6-20', '21-100', '101-500', '500+']
df_summary['review_bucket'] = pd.cut(df_summary[PSUM_REVIEW_COUNT], bins=bins, labels=labels)

bucket_stats = (
    df_summary.groupby('review_bucket', observed=True)
    .agg(
        product_count=('review_bucket', 'size'),
        avg_rating=(PSUM_AVG_RATING, 'mean'),
        median_rating=(PSUM_AVG_RATING, 'median')
    )
    .reset_index()
)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Avg Rating by Review Count', 'Products per Bucket'],
                    horizontal_spacing=0.12)

fig.add_trace(go.Bar(
    x=bucket_stats['review_bucket'], y=bucket_stats['avg_rating'],
    marker_color='#FF9800',
    text=[f'{v:.2f}' for v in bucket_stats['avg_rating']],
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=bucket_stats['review_bucket'], y=bucket_stats['product_count'],
    marker_color='#607D8B'
), row=1, col=2)

fig.update_layout(title='Review Volume vs Rating — Do More Reviews Mean Lower Ratings?',
                  template=TEMPLATE, height=500, showlegend=False)
fig.update_yaxes(title_text='Avg Rating', row=1, col=1)
fig.update_yaxes(title_text='# Products', row=1, col=2)
save_chart(fig, '11_review_buckets_vs_rating')
fig.show()

---

## 7 — Temporal Review Patterns

Finding #35: Ratings declining 4.28 → 4.02 (2019–2022).  
Finding #36: 2023 review volume drops (partial dataset).  
How do image usage, helpfulness, and verification change over time?

In [29]:
TEMP_YEAR = 'review_year'

yearly = (
    df_temporal.groupby(TEMP_YEAR)
    .agg(
        total_reviews=('review_count', 'sum'),
        avg_rating=('avg_rating', 'mean'),
        pct_verified=('pct_verified', 'mean'),
        pct_with_images=('pct_with_images', 'mean'),
        avg_helpful=('avg_helpful_votes', 'mean'),
        avg_length=('avg_text_length', 'mean')
    )
    .reset_index()
)
yearly = yearly[(yearly[TEMP_YEAR] >= 2010) & (yearly[TEMP_YEAR] <= 2023)]

metrics = [
    ('pct_verified', '% Verified', '#4CAF50'),
    ('pct_with_images', '% With Images', '#2196F3'),
    ('avg_helpful', 'Avg Helpful Votes', '#FF9800'),
    ('avg_length', 'Avg Text Length', '#9C27B0')
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics],
                    horizontal_spacing=0.1, vertical_spacing=0.12)

for idx, (col, label, color) in enumerate(metrics):
    row, c = idx // 2 + 1, idx % 2 + 1
    fig.add_trace(go.Scatter(
        x=yearly[TEMP_YEAR], y=yearly[col],
        mode='lines+markers', line=dict(color=color, width=3),
        name=label, showlegend=False
    ), row=row, col=c)

fig.update_layout(title='Review Behavior Evolution (2010–2023)',
                  template=TEMPLATE, height=700)
save_chart(fig, '12_review_behavior_over_time')
fig.show()

---

## 8 — Key Findings

In [32]:
print('=' * 60)
print('REVIEWS & SENTIMENT — KEY FINDINGS')
print('=' * 60)

print(f'\n1. SENTIMENT LANDSCAPE:')
worst = df_sent.nlargest(3, SENT_PCT_NEG)
for _, row in worst.iterrows():
    print(f'   {row[SENT_CAT_COL]}: {row[SENT_PCT_NEG]:.1f}% negative')
best = df_sent.nsmallest(3, SENT_PCT_NEG)
for _, row in best.iterrows():
    print(f'   {row[SENT_CAT_COL]}: {row[SENT_PCT_NEG]:.1f}% negative')

print(f'\n2. VERBOSITY:')
print(f'   Avg verbosity ratio: {df_sent[SENT_VERBOSITY].mean():.2f}×')
print(f'   Most verbose complaints: {df_sent.nlargest(1, SENT_VERBOSITY)[SENT_CAT_COL].values[0]} '
      f'({df_sent[SENT_VERBOSITY].max():.2f}×)')

print(f'\n3. COMPLAINT KEYWORDS (distinctive):')
top5_neg = overall_neg.nlargest(5, 'neg_ratio')
for _, row in top5_neg.iterrows():
    print(f'   {row[KW_WORD]}: {row["neg_ratio"]:.0f}× more in negative ({row["total_neg"]:,} uses)')

print(f'\n4. PRAISE KEYWORDS (distinctive):')
top5_pos = overall_pos.nlargest(5, 'pos_ratio')
for _, row in top5_pos.iterrows():
    print(f'   {row[KW_WORD]}: {row["pos_ratio"]:.0f}× more in positive ({row["total_pos"]:,} uses)')

print(f'\n5. REVIEW TRUST:')
print(f'   Overall verified rate: {df_trust_cat[TRUST_VERIFIED_PCT].mean():.1f}%')
print(f'   Lowest verified: {df_trust_cat.nsmallest(1, TRUST_VERIFIED_PCT)[TRUST_CAT_COL].values[0]} '
      f'({df_trust_cat[TRUST_VERIFIED_PCT].min():.1f}%)')
print(f'   Highest verified: {df_trust_cat.nlargest(1, TRUST_VERIFIED_PCT)[TRUST_CAT_COL].values[0]} '
      f'({df_trust_cat[TRUST_VERIFIED_PCT].max():.1f}%)')
gap_mean = df_trust_cat['unverified_5star_gap'].mean()
print(f'   Avg unverified 5-star gap: {gap_mean:+.1f}pp (negative = unverified LESS generous)')

print(f'\n6. PRODUCT REVIEW PATTERNS:')
print(f'   Products with reviews: {len(df_summary):,} of 1.4M ({len(df_summary)/1_426_337:.1%})')
print(f'   Median reviews/product: {df_summary[PSUM_REVIEW_COUNT].median():.0f}')
print(f'   Mean reviews/product: {df_summary[PSUM_REVIEW_COUNT].mean():.0f}')

print(f'\n7. KEYWORDS:')
print(f'   Unique keywords tracked: {df_kw[KW_WORD].nunique():,}')
print(f'   Categories covered: {df_kw[KW_CAT_COL].nunique()}')

REVIEWS & SENTIMENT — KEY FINDINGS

1. SENTIMENT LANDSCAPE:
   Subscription_Boxes: 25.1% negative
   All_Beauty: 20.7% negative
   Cell_Phones_and_Accessories: 19.5% negative
   Kindle_Store: 6.1% negative
   CDs_and_Vinyl: 6.8% negative
   Digital_Music: 7.1% negative

2. VERBOSITY:
   Avg verbosity ratio: 1.33×
   Most verbose complaints: Gift_Cards (2.98×)

3. COMPLAINT KEYWORDS (distinctive):
   inedible: 699× more in negative (699 uses)
   flavorless: 608× more in negative (608 uses)
   rancid: 396× more in negative (1,586 uses)
   dnf: 383× more in negative (766 uses)
   dissatisfied: 336× more in negative (5,375 uses)

4. PRAISE KEYWORDS (distinctive):
   inspiring: 4229× more in positive (12,686 uses)
   captivating: 3851× more in positive (11,552 uses)
   steamy: 3727× more in positive (3,727 uses)
   yum: 3555× more in positive (17,777 uses)
   insightful: 3323× more in positive (6,646 uses)

5. REVIEW TRUST:
   Overall verified rate: 89.2%
   Lowest verified: CDs_and_Vinyl (

In [31]:
con.close()
print('Done. DuckDB connection closed.')
print(f'Charts saved to: {CHARTS_DIR}/')

Done. DuckDB connection closed.
Charts saved to: charts/07_reviews_sentiment/
